# Notebook 03 — Experimentos Deep Learning — Wavelet Fixo db2 (FordA)

Classificação binária com modelos DL usando coeficientes wavelet fixos (db2) como entrada multi-canal.
- **CNN + Wavelet**
- **LSTM + Wavelet**
- **CNN-LSTM + Wavelet**
- **Transformer + Wavelet**

**Pipeline:** Sinal → Wavelet db2 (approx + details como canais) → Grid Search → Avaliação

In [ ]:
import os, sys
sys.path.append(".")
from config.experiment_config import (
    DATA_DIR, RESULTS_DIR, MODELS_DIR,
    DL_TRAINING_CONFIG, DL_MODELS_CONFIG, WAVELET_CONFIG,
    generate_dl_grid, SEED, GPU_ID, EPOCHS_OVERRIDE, MAX_GRID_CONFIGS,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import random
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
# Limitar a 1 GPU para evitar problemas de batch com MirroredStrategy
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.set_visible_devices([gpus[0]], 'GPU')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

from src.models import (
    create_cnn_model, create_lstm_model,
    create_cnn_lstm_model, create_transformer_model,
    get_callbacks, get_distribute_strategy,
)
from src.feature_extraction import WaveletFeatureExtractor
from src.evaluation import ClassificationEvaluator, ResultsManager
from src.visualization import ExperimentVisualizer

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

strategy = get_distribute_strategy()

DL_WAV_DIR = RESULTS_DIR / "dl_wavelet_experiments"
DL_WAV_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nSEED={SEED}")
if GPU_ID: print(f"GPU selecionada: {GPU_ID}")
if EPOCHS_OVERRIDE: print(f"EPOCHS_OVERRIDE={EPOCHS_OVERRIDE}")
if MAX_GRID_CONFIGS: print(f"MAX_GRID_CONFIGS={MAX_GRID_CONFIGS}")

## 1. Carregar e Transformar Dados

In [ ]:
X_train_raw = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_val_raw = np.load(DATA_DIR / "X_val.npy")
y_val = np.load(DATA_DIR / "y_val.npy")
X_test_raw = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

print(f"Dados Raw:")
print(f"  Train: {X_train_raw.shape}, Val: {X_val_raw.shape}, Test: {X_test_raw.shape}")

In [ ]:
# Transformar para coeficientes wavelet multi-nível (como canais)
wfe = WaveletFeatureExtractor(
    wavelet=WAVELET_CONFIG["wavelet_type"],
    level=WAVELET_CONFIG["decomposition_level"],
    mode=WAVELET_CONFIG["mode"],
)

print(f"Wavelet: {WAVELET_CONFIG['wavelet_type']}, Níveis: {WAVELET_CONFIG['decomposition_level']}")
print("\nTransformando sinais para coeficientes wavelet...")
t0 = time.time()

X_train = wfe.get_multilevel_coefficients(X_train_raw, align_length=True)
X_val = wfe.get_multilevel_coefficients(X_val_raw, align_length=True)
X_test = wfe.get_multilevel_coefficients(X_test_raw, align_length=True)

print(f"  Tempo: {time.time()-t0:.2f}s")
print(f"\nDados Transformados (Wavelet Coefficients):")
print(f"  Train: {X_train.shape}  [samples, length, channels=níveis+1]")
print(f"  Val:   {X_val.shape}")
print(f"  Test:  {X_test.shape}")

channel_names = ["Approx"] + [f"Detail_{i}" for i in range(1, WAVELET_CONFIG["decomposition_level"]+1)]
print(f"  Canais: {channel_names}")

input_shape = X_train.shape[1:]
print(f"\nInput shape para modelos: {input_shape}")

## 2. Visualização dos Coeficientes Wavelet

In [ ]:
fig, axes = plt.subplots(len(channel_names) + 1, 1, figsize=(14, 3 * (len(channel_names) + 1)))
idx = 0

axes[0].plot(X_train_raw[idx])
axes[0].set_title("Sinal Original")
axes[0].set_ylabel("Amplitude")

for ch, name in enumerate(channel_names):
    axes[ch + 1].plot(X_train[idx, :, ch])
    axes[ch + 1].set_title(f"Canal: {name}")
    axes[ch + 1].set_ylabel("Coef.")

axes[-1].set_xlabel("Tempo")
plt.tight_layout()
plt.savefig(DL_WAV_DIR / "wavelet_channels_sample.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Configuração

In [ ]:
results_manager = ResultsManager(DL_WAV_DIR)
evaluator = ClassificationEvaluator()
viz = ExperimentVisualizer()

training_config = DL_TRAINING_CONFIG.copy()
print("Configuração de Treinamento:")
for k, v in training_config.items():
    print(f"  {k}: {v}")

all_results = {}
all_histories = {}
all_grid_results = []

## 4. Experimento 1: CNN + Wavelet

In [ ]:
print("=" * 70)
print("Grid Search: CNN com Wavelet db2")
print("=" * 70)

grid = generate_dl_grid("CNN")
base_params = DL_MODELS_CONFIG["CNN"].copy()
best_acc, best_key = 0.0, None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"Wav_CNN_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_cnn_model(input_shape, params=params)

    model_path = str(MODELS_DIR / f"wav_cnn_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config["early_stopping_patience"],
        patience_lr=training_config["reduce_lr_patience"],
        min_lr=training_config["min_lr"],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config["epochs"],
        batch_size=training_config["batch_size"],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"    Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f}  Time={elapsed:.1f}s")

    row = {"Model": "Wav_CNN", "grid_idx": gi, **variation,
           "Accuracy": metrics["accuracy"], "F1": metrics["f1"],
           "Precision": metrics["precision"], "Recall": metrics["recall"],
           "AUC_ROC": metrics["auc_roc"], "Params": model.count_params(),
           "Time (s)": round(elapsed, 1), "Epochs": len(history.history["loss"])}
    all_grid_results.append(row)

    if metrics["accuracy"] > best_acc:
        best_acc = metrics["accuracy"]
        best_key = run_name
        all_results["Wav_CNN"] = {
            "metrics": metrics, "time": elapsed,
            "epochs": len(history.history["loss"]),
            "y_pred": y_pred, "y_prob": y_prob,
            "params": model.count_params(),
            "best_variation": variation,
        }
        all_histories["Wav_CNN"] = history.history
        model.save(str(MODELS_DIR / "wav_cnn_best.keras"))

    results_manager.log_experiment("DL_Wavelet", run_name, metrics, {"params": params})

print(f"\nMelhor CNN: {best_key} — Acc={best_acc:.4f}")


## 5. Experimento 2: LSTM + Wavelet

In [ ]:
print("=" * 70)
print("Grid Search: LSTM com Wavelet db2")
print("=" * 70)

grid = generate_dl_grid("LSTM")
base_params = DL_MODELS_CONFIG["LSTM"].copy()
best_acc, best_key = 0.0, None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"Wav_LSTM_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_lstm_model(input_shape, params=params)

    model_path = str(MODELS_DIR / f"wav_lstm_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config["early_stopping_patience"],
        patience_lr=training_config["reduce_lr_patience"],
        min_lr=training_config["min_lr"],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config["epochs"],
        batch_size=training_config["batch_size"],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"    Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f}  Time={elapsed:.1f}s")

    row = {"Model": "Wav_LSTM", "grid_idx": gi, **variation,
           "Accuracy": metrics["accuracy"], "F1": metrics["f1"],
           "Precision": metrics["precision"], "Recall": metrics["recall"],
           "AUC_ROC": metrics["auc_roc"], "Params": model.count_params(),
           "Time (s)": round(elapsed, 1), "Epochs": len(history.history["loss"])}
    all_grid_results.append(row)

    if metrics["accuracy"] > best_acc:
        best_acc = metrics["accuracy"]
        best_key = run_name
        all_results["Wav_LSTM"] = {
            "metrics": metrics, "time": elapsed,
            "epochs": len(history.history["loss"]),
            "y_pred": y_pred, "y_prob": y_prob,
            "params": model.count_params(),
            "best_variation": variation,
        }
        all_histories["Wav_LSTM"] = history.history
        model.save(str(MODELS_DIR / "wav_lstm_best.keras"))

    results_manager.log_experiment("DL_Wavelet", run_name, metrics, {"params": params})

print(f"\nMelhor LSTM: {best_key} — Acc={best_acc:.4f}")


## 6. Experimento 3: CNN_LSTM + Wavelet

In [ ]:
print("=" * 70)
print("Grid Search: CNN_LSTM com Wavelet db2")
print("=" * 70)

grid = generate_dl_grid("CNN_LSTM")
base_params = DL_MODELS_CONFIG["CNN_LSTM"].copy()
best_acc, best_key = 0.0, None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"Wav_CNN_LSTM_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_cnn_lstm_model(input_shape, params=params)

    model_path = str(MODELS_DIR / f"wav_cnn_lstm_g{gi}.keras")
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config["early_stopping_patience"],
        patience_lr=training_config["reduce_lr_patience"],
        min_lr=training_config["min_lr"],
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config["epochs"],
        batch_size=training_config["batch_size"],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"    Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f}  Time={elapsed:.1f}s")

    row = {"Model": "Wav_CNN_LSTM", "grid_idx": gi, **variation,
           "Accuracy": metrics["accuracy"], "F1": metrics["f1"],
           "Precision": metrics["precision"], "Recall": metrics["recall"],
           "AUC_ROC": metrics["auc_roc"], "Params": model.count_params(),
           "Time (s)": round(elapsed, 1), "Epochs": len(history.history["loss"])}
    all_grid_results.append(row)

    if metrics["accuracy"] > best_acc:
        best_acc = metrics["accuracy"]
        best_key = run_name
        all_results["Wav_CNN_LSTM"] = {
            "metrics": metrics, "time": elapsed,
            "epochs": len(history.history["loss"]),
            "y_pred": y_pred, "y_prob": y_prob,
            "params": model.count_params(),
            "best_variation": variation,
        }
        all_histories["Wav_CNN_LSTM"] = history.history
        model.save(str(MODELS_DIR / "wav_cnn_lstm_best.keras"))

    results_manager.log_experiment("DL_Wavelet", run_name, metrics, {"params": params})

print(f"\nMelhor CNN_LSTM: {best_key} — Acc={best_acc:.4f}")


## 7. Experimento 4: Transformer + Wavelet

In [ ]:
print("=" * 70)
print("Grid Search: Transformer com Wavelet db2")
print("=" * 70)

grid = generate_dl_grid("Transformer")
base_params = DL_MODELS_CONFIG["Transformer"].copy()
best_acc, best_key = 0.0, None

for gi, variation in enumerate(grid):
    params = {**base_params, **variation}
    run_name = f"Wav_Transformer_g{gi}"
    print(f"\n--- [{gi+1}/{len(grid)}] {run_name}: {variation}")

    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_transformer_model(input_shape, params=params)

    model_path = str(MODELS_DIR / f"wav_transformer_g{gi}.keras")
    # Transformer usa WarmupSchedule, incompatível com ReduceLROnPlateau
    callbacks = get_callbacks(
        model_path,
        patience_early=training_config["early_stopping_patience"],
        use_reduce_lr=False,
    )

    t0 = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=training_config["epochs"],
        batch_size=training_config["batch_size"],
        callbacks=callbacks, verbose=0,
    )
    elapsed = time.time() - t0

    y_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    metrics = evaluator.evaluate(y_test, y_pred, y_prob)

    print(f"    Acc={metrics['accuracy']:.4f}  F1={metrics['f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f}  Time={elapsed:.1f}s")

    row = {"Model": "Wav_Transformer", "grid_idx": gi, **variation,
           "Accuracy": metrics["accuracy"], "F1": metrics["f1"],
           "Precision": metrics["precision"], "Recall": metrics["recall"],
           "AUC_ROC": metrics["auc_roc"], "Params": model.count_params(),
           "Time (s)": round(elapsed, 1), "Epochs": len(history.history["loss"])}
    all_grid_results.append(row)

    if metrics["accuracy"] > best_acc:
        best_acc = metrics["accuracy"]
        best_key = run_name
        all_results["Wav_Transformer"] = {
            "metrics": metrics, "time": elapsed,
            "epochs": len(history.history["loss"]),
            "y_pred": y_pred, "y_prob": y_prob,
            "params": model.count_params(),
            "best_variation": variation,
        }
        all_histories["Wav_Transformer"] = history.history
        model.save(str(MODELS_DIR / "wav_transformer_best.keras"))

    results_manager.log_experiment("DL_Wavelet", run_name, metrics, {"params": params})

print(f"\nMelhor Transformer: {best_key} — Acc={best_acc:.4f}")

## 8. Comparação dos Resultados

In [ ]:
comp_rows = []
for name, info in all_results.items():
    row = {"Model": name, **info["metrics"],
           "Params": info["params"], "Time (s)": round(info["time"], 1),
           "Epochs": info["epochs"]}
    comp_rows.append(row)

comparison_df = pd.DataFrame(comp_rows).sort_values("accuracy", ascending=False)
comparison_df = comparison_df.set_index("Model")
print("\n" + "=" * 80)
print("COMPARAÇÃO DOS MODELOS DL (Wavelet db2)")
print("=" * 80)
print(comparison_df.to_string())

comparison_df.to_csv(DL_WAV_DIR / "comparison_dl_wavelet.csv")
pd.DataFrame(all_grid_results).to_csv(DL_WAV_DIR / "all_grid_results_dl_wavelet.csv", index=False)
print(f"\nResultados salvos em {DL_WAV_DIR}")

In [ ]:
metrics_to_plot = ["accuracy", "f1", "auc_roc"]
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(6 * len(metrics_to_plot), 5))
model_names = comparison_df.index.tolist()
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(model_names)))

for ax, metric in zip(axes, metrics_to_plot):
    values = comparison_df[metric].values
    bars = ax.barh(range(len(model_names)), values, color=colors)
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels(model_names)
    ax.set_xlabel(metric.upper())
    ax.set_title(metric.upper())
    ax.invert_yaxis()
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
fig.suptitle("Comparação DL Wavelet db2 — FordA", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(DL_WAV_DIR / "comparison_chart_dl_wavelet.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Evolução do Treinamento

In [ ]:
n_models = len(all_histories)
if n_models > 0:
    fig, axes = plt.subplots(n_models, 2, figsize=(14, 4 * n_models))
    if n_models == 1:
        axes = axes.reshape(1, -1)
    for i, (name, hist) in enumerate(all_histories.items()):
        axes[i, 0].plot(hist["loss"], label="Train")
        axes[i, 0].plot(hist["val_loss"], label="Val")
        axes[i, 0].set_title(f"{name} — Loss")
        axes[i, 0].set_xlabel("Epoch")
        axes[i, 0].legend()
        acc_key = "accuracy" if "accuracy" in hist else "acc"
        val_acc_key = "val_accuracy" if "val_accuracy" in hist else "val_acc"
        if acc_key in hist:
            axes[i, 1].plot(hist[acc_key], label="Train")
            axes[i, 1].plot(hist[val_acc_key], label="Val")
            axes[i, 1].set_title(f"{name} — Accuracy")
            axes[i, 1].set_xlabel("Epoch")
            axes[i, 1].legend()
    plt.tight_layout()
    plt.savefig(DL_WAV_DIR / "training_evolution_dl_wavelet.png", dpi=150, bbox_inches="tight")
    plt.show()

## 10. Confusion Matrices e ROC

In [ ]:
from sklearn.metrics import roc_curve, auc

for name, info in all_results.items():
    viz.plot_confusion_matrix(
        y_test, info["y_pred"],
        model_name=name,
        save_path=DL_WAV_DIR / f"confusion_matrix_{name}.png",
    )
    plt.show()

roc_data = {}
for name, info in all_results.items():
    if "y_prob" in info:
        fpr, tpr, _ = roc_curve(y_test, info["y_prob"])
        auc_val = auc(fpr, tpr)
        roc_data[name] = (fpr, tpr, auc_val)
if roc_data:
    viz.plot_multi_roc(
        roc_data,
        save_path=DL_WAV_DIR / "roc_curves_dl_wavelet.png",
    )
    plt.show()

## 11. Resumo

In [ ]:
print("=" * 60)
print("RESUMO — DL Wavelet db2 Experiments (FordA)")
print("=" * 60)
print(f"Wavelet: {WAVELET_CONFIG['wavelet_type']}, Níveis: {WAVELET_CONFIG['decomposition_level']}")
print(f"Input shape: {input_shape}")
print(f"Arquiteturas avaliadas: {len(all_results)}")
print(f"Total de configurações grid: {len(all_grid_results)}")
if all_results:
    best_name = comparison_df.index[0]
    print(f"Melhor modelo: {best_name}")
    print(f"  Accuracy: {comparison_df.loc[best_name, 'accuracy']:.4f}")
    print(f"  F1:       {comparison_df.loc[best_name, 'f1']:.4f}")
    print(f"  AUC-ROC:  {comparison_df.loc[best_name, 'auc_roc']:.4f}")
print(f"Resultados em: {DL_WAV_DIR}")